In [20]:
import pandas as pd

alert_threshold = 30

df_validation = pd.read_csv("../outputs/VALIDATION_FD001.csv")
df_fitting = pd.read_csv("../outputs/FITTING_FD001.csv")

df_validation["ALERT"] = 0
df_fitting["ALERT"] = 0

df_validation.loc[df_validation["RUL"] <= alert_threshold, "ALERT"] = 1
df_fitting.loc[df_fitting["RUL"] <= alert_threshold, "ALERT"] = 1


In [21]:
print(df_validation["ALERT"].value_counts(normalize=True))
print(df_fitting["ALERT"].value_counts(normalize=True))

ALERT
0    0.840576
1    0.159424
Name: proportion, dtype: float64
ALERT
0    0.85187
1    0.14813
Name: proportion, dtype: float64


we can see that with an alert threshold of 30 cycles RUL - we have a distribution of around 85% no alert and 15% alert. Now we are gonna implement a dummy classifier that simply predict the most common classification (in my case "NO ALERT" (0))

In [22]:
x_fitting = df_fitting.iloc[:, 1:26]
y_fitting = df_fitting["ALERT"]

x_validation = df_validation.iloc[:, 1:26]
y_validation = df_validation["ALERT"]

In [23]:
from sklearn.dummy import DummyClassifier
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(x_fitting, y_fitting)

predictions = dummy_model.predict(x_validation)

print (predictions)

[0 0 0 ... 0 0 0]


In [24]:
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, precision_score

c_matrix = confusion_matrix(y_validation, predictions)
print(c_matrix)

[[3269    0]
 [ 620    0]]


From the confusion matrix we see that we have 3269 correctly detected non-alarms, 620 missed alarms, 0 false alarms and 0 deteccted alarms.
Now we can already see a common mistake: judging the performance of a classifier based on the accuracy score: 

In [25]:
ac_score = accuracy_score(y_validation, predictions)
print(ac_score)

0.8405759835433273


If I decrease the alert_threshold to 5 cycles - then we have barely any alarm labels. And therefore the accuracy score will make it seem like our model is performing very good (96,91% correctness) even tho its just a stupid model going for that classification that is most common. For that it can make sense to verify with the recall score:

In [26]:
re_score = recall_score(y_validation, predictions)
print(re_score)

0.0


the precision:

In [27]:
pr_score = precision_score(y_validation, predictions, zero_division=0)
print(pr_score)

0.0


lastly the F1 score combines precision (of all predicted alarms, what fraction was correct) and recall (of all actual alarm cases, what fraction was detected?)

In [28]:
f1 = f1_score(y_validation, predictions)
print(f1)

0.0


Now instead of using the dummy classifier, we will use logistic regression to predict wether there is an ALARM or not.

In [29]:
probability_threshold = 0.5

# check mean and std before we standardize

print("Mean of x_fitting:")
print(x_fitting.mean())

print("Std of x_fitting:")
print(x_fitting.std())

Mean of x_fitting:
1     1.106507e+02
2    -1.317644e-05
3    -8.720583e-07
4     1.000000e+02
5     5.186700e+02
6     6.426817e+02
7     1.590504e+03
8     1.408942e+03
9     1.462000e+01
10    2.160980e+01
11    5.533645e+02
12    2.388097e+03
13    9.064970e+03
14    1.300000e+00
15    4.754173e+01
16    5.214120e+02
17    2.388097e+03
18    8.143523e+03
19    8.442179e+00
20    3.000000e-02
21    3.932131e+02
22    2.388000e+03
23    1.000000e+02
24    3.881631e+01
25    2.328972e+01
dtype: float64
Std of x_fitting:
1     7.024753e+01
2     2.192971e-03
3     2.932231e-04
4     0.000000e+00
5     1.136902e-13
6     5.009118e-01
7     6.140107e+00
8     9.002827e+00
9     3.552820e-15
10    1.396268e-03
11    8.845205e-01
12    7.116298e-02
13    2.218064e+01
14    0.000000e+00
15    2.671468e-01
16    7.392389e-01
17    7.218095e-02
18    1.918481e+01
19    3.747185e-02
20    1.040865e-17
21    1.548550e+00
22    0.000000e+00
23    0.000000e+00
24    1.803678e-01
25    1.079854e-0

We can see that the mean and standard derivations on the sensors are of quite different scales. That makes it more difficult for the logistic regression model to pick weights. Therefore in the following a simple standardization is applied:

In [30]:
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
scaler.fit(x_fitting)
scaled_x_fitting = scaler.transform(x_fitting)

scaled_x_validation = scaler.transform(x_validation)

In [31]:
from sklearn import linear_model

log_reg_model = linear_model.LogisticRegression()
log_reg_model.fit(scaled_x_fitting, y_fitting)

log_reg_predictions = log_reg_model.predict(scaled_x_validation)

c_matrix = confusion_matrix(y_validation, log_reg_predictions)
print(c_matrix)


[[3211   58]
 [ 118  502]]


Finally all the performance scores:

In [32]:
ac_score = accuracy_score(y_validation, log_reg_predictions)
print("ac_score: ", ac_score)

re_score = recall_score(y_validation, log_reg_predictions)
print("recall_score: ", re_score)

pr_score = precision_score(y_validation, log_reg_predictions)
print("precision_score: ", pr_score)

f1 = f1_score(y_validation, log_reg_predictions)
print("f1_score: ", f1)

ac_score:  0.9547441501671381
recall_score:  0.8096774193548387
precision_score:  0.8964285714285715
f1_score:  0.8508474576271187


The current predict function uses a confidence of 50%. Meaning it classifies as ALARM when it estimates 50% or more likelihood of ALARM. 

That makes sense in many scenarios. But in some we want to make sure (almost) no Maintenance Alarms go unnoticed. If we decrease the confidence to for example 10%, we expect a much higher recall, which is exactly what we want because we really dont want to run into a situation where an engine failes because we didnt provide maintenance because of a missed alarm.

In [33]:
confidence_threshold = 0.1

# check class order
print(log_reg_model.classes_)

log_reg_probas = log_reg_model.predict_proba(scaled_x_validation)
df_validation["proba"] = log_reg_probas[:, 1]

df_validation["proba_predict"] = 0
df_validation.loc[df_validation["proba"] >= confidence_threshold , "proba_predict"] = 1


[0 1]


In [34]:
c_matrix2 = confusion_matrix(y_validation, df_validation["proba_predict"])
print(c_matrix2)

[[3063  206]
 [  26  594]]


We can see that we have 206 False alarms instead of 58 (for 50% cutoff).  
As a tradeoff we have just 26 missed alerts instead of 118. That can be very important and what's better totally depends on the system of concern.

In [35]:
ac_score = accuracy_score(y_validation, df_validation["proba_predict"])
print("ac_score: ", ac_score)

re_score = recall_score(y_validation, df_validation["proba_predict"])
print("recall_score: ", re_score)

pr_score = precision_score(y_validation, df_validation["proba_predict"])
print("precision_score: ", pr_score)

f1 = f1_score(y_validation, df_validation["proba_predict"])
print("f1_score: ", f1)

ac_score:  0.9403445615839547
recall_score:  0.9580645161290322
precision_score:  0.7425
f1_score:  0.8366197183098592


As a result also the recall increased which was our goal. As a tradeoff all the other scores are lower.

### Decision Tree Classifier

In [36]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(min_samples_leaf=100, max_depth=5)
tree_model.fit(x_fitting, y_fitting)
tree_predictions = tree_model.predict(x_validation)

In [37]:
c_matrix3 = confusion_matrix(y_validation, tree_predictions)
print(c_matrix3)
print()

ac_score = accuracy_score(y_validation, tree_predictions)
print("ac_score: ", ac_score)

re_score = recall_score(y_validation, tree_predictions)
print("recall_score: ", re_score)

pr_score = precision_score(y_validation, tree_predictions)
print("precision_score: ", pr_score)

f1 = f1_score(y_validation, tree_predictions)
print("f1_score: ", f1)

[[3224   45]
 [ 168  452]]

ac_score:  0.9452301362818205
recall_score:  0.7290322580645161
precision_score:  0.9094567404426559
f1_score:  0.8093106535362579


I'll mark down the wrong guesses throughout increasing depth in the tree.  
depth = 1: 99x False Alarm, 171x Missed Alarm.  
depth = 2: 113x False Alarm, 111x Missed Alarm.  
depth = 3: 38x False Alarm, 190x Missed Alarm.  
depth = 4: 35x False Alarm, 182x Missed Alarm.  
depth = 5: 41x False Alarm, 159x Missed Alarm.  
depth = 10: 74x False Alarm, 124x Missed Alarm.

Further increasing or changing other parameters doesnt help much. Decision Tree Classifier therefore didnt perform as good as the logistic regression on this dataset.

A very important thing is to check for overfitting (in general). One way to do is to compare scores (I will compare F1), for predictions on the FITTING data with the one on the Validation data. If they are very far apart with the fitting data performing a lot better - we can assume some overfitting took place.

In [38]:
tree_predictions_on_fitting = tree_model.predict(x_fitting)

print(f1_score(y_validation, tree_predictions))
print(f1_score(y_fitting, tree_predictions_on_fitting))

0.8093106535362579
0.84593837535014


### Results summary

Validation cycles; alert label: `RUL <= 30`. Precision, recall and F1 refer to class 1.

| Classifier | Settings | Accuracy | Precision | Recall | F1 | False alarms | Missed alerts |
| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Dummy | Most frequent class | 84.06% | 0.00%* | 0.00% | 0.00% | 0 | 620 |
| Logistic regression | Standardized inputs; default cutoff (~0.50) | 95.47% | 89.64% | 80.97% | 85.08% | 58 | 118 |
| Logistic regression | Standardized inputs; cutoff 0.10 | 94.03% | 74.25% | 95.81% | 83.66% | 206 | 26 |
| Decision tree | Depth 5; minimum leaf size 100 | 94.52% | 90.95% | 72.90% | 80.93% | 45 | 168 |

*Dummy precision is undefined; reported as zero with `zero_division=0`. Counts refer to cycles.

Tree-depth results from experiment notes (other settings not fully recorded):

| Maximum depth | False alarms | Missed alerts |
| --- | ---: | ---: |
| 1 | 99 | 171 |
| 2 | 113 | 111 |
| 3 | 38 | 190 |
| 4 | 35 | 182 |
| 5 | 41 | 159 |
| 10 | 74 | 124 |

Saved tree F1: fitting **84.59%**, validation **80.93%**.

*Table compilation: Codex, from existing saved results and notes.*